# Open Insect ID v2 — Kaggle pipeline
#
# This notebook is the runnable, on-Kaggle entry point. It installs the
# `src` package from GitHub and exercises the full data pipeline. Set the
# accelerator to **CPU** for the data stages (GPU is not needed) and use the
# training notebook (02) with a **GPU**.
#
# Fill in `GITHUB_REPO` below with your repository URL (or use the local copy
# already in the Kaggle notebook's working directory).


## 1. Setup & configuration

In [ ]:
import os, subprocess, sys

# --- Configure this to your repo -------------------------------------------
GITHUB_REPO = os.environ.get("OPENINSECT_REPO", "https://github.com/YoannDev90/OpenInsectID.git")
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE and "OpenInsectID" not in os.getcwd():
    # Install the package straight from GitHub on a fresh Kaggle kernel.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", GITHUB_REPO])
else:
    # Local dev: ensure the repo root is importable.
    repo_root = os.getcwd()
    if not os.path.isdir(os.path.join(repo_root, "src")):
        repo_root = str(Path(GITHUB_REPO.split(".git")[0]).parent)
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

from pathlib import Path
import yaml

print("package ready; importable as `src.*`")


In [ ]:
# Load the data pipeline configuration (filters, H3 resolution, curation).
from src.data_pipeline import GBIFConfig, H3Config, CurationConfig, DownloadConfig

gbif_cfg = GBIFConfig(
    class_key=216,               # Insecta
    media_type="StillImage",
    occurrence_status="PRESENT",
    basis_of_record=("HUMAN_OBSERVATION", "PRESERVED_SPECIMEN"),
    has_coordinate=True,
)
h3_cfg = H3Config(resolution=4, min_occurrences_per_cell=3)
cur_cfg = CurationConfig(min_resolution=224, max_aspect_ratio=3.0)
dl_cfg = DownloadConfig(concurrency=8, per_domain_rps=4.0, raw_root="data/raw")
print("Insecta, world-scale, H3 res", h3_cfg.resolution)


## 2. Ingest occurrences (GBIF)

Two interchangeable sources:
- **A. Async API** (`GBIFExtractor`) — best for a targeted slice / prototyping.
- **B. DuckDB over a bulk TSV** (`SpatialProcessor.load_occurrences`) — best for
  the full world-scale dump (tens of millions of rows) streamed off disk.


In [ ]:
# --- Option A: async GBIF occurrence search (prototype / bounded slice) ------
from src.data_pipeline import GBIFExtractor

extractor = GBIFExtractor(gbif_cfg)
# Limit records for a quick pass; drop `max_records=None` to page the full set.
records = extractor.extract_sync(max_records=500)
print(f"fetched {len(records)} image-bearing occurrences")
for r in records[:3]:
    print(" ", r.lineage.order, "/", r.lineage.family, "/", r.lineage.genus, "/", r.lineage.species)


In [ ]:
# --- Option B: spatial clean + H3 range mask (world-scale TSV) --------------
# Point `tsv_path` at a GBIF occurrence dump (TSV/CSV) if you have one.
from src.data_pipeline import SpatialProcessor

# sp = SpatialProcessor(h3_cfg)
# occ = sp.load_occurrences("data/gbif_insecta.tsv")   # streams via DuckDB
# clean, stats = sp.clean(occ)                          # builds H3 mask + flags outliers
# print(stats)
print("Uncomment to run DuckDB spatial cleaning on a full GBIF dump.")


## 3. Download images (async, fault-tolerant)

In [ ]:
from src.data_pipeline import ImageDownloader

downloader = ImageDownloader(dl_cfg)
# result = downloader.download_sync(records)
# print(f"ok={result.ok} failed={result.failed} skipped(dup)={result.skipped}")
print("Uncomment to download images into data/raw/{order}/{family}/{genus}/{species}/")


## 4. Curation (integrity / resolution / aspect ratio)

In [ ]:
from src.data_pipeline import curate_directory

# verdicts = curate_directory("data/raw", cur_cfg)
# ok = sum(v.ok for v in verdicts)
# print(f"{ok}/{len(verdicts)} images pass curation")
# for v in verdicts:
#     if not v.ok:
#         print("  drop:", v.path, "->", v.reason)
print("Uncomment to run automated curation over data/raw.")


## 5. Export the curated manifest

Write a Parquet manifest consumed by the training notebook. Schema:
`image_path, order, family, genus, species` (+ optional `h3_cell, lat, lon`).
On Kaggle, push this as a Dataset so notebook 02 can attach it as input.


In [ ]:
import polars as pl

# Build a manifest DataFrame from your curated records, then:
# manifest.write_parquet("insectid_manifest.parquet")
# If running on Kaggle, publish with:
#   !kaggle datasets create -p . --dir-mode tar
# and attach it as an input dataset in notebook 02.
print("Manifest schema:", ["image_path", "order", "family", "genus", "species"])
